# Práctica: Redes Convolucionales

## 0. Importación de librerías

In [ ]:

import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from PIL import Image

from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix

import tensorflow as tf
from tensorflow.keras import layers, models
from tensorflow.keras.callbacks import EarlyStopping


## 1. Lectura de imágenes y creación de datasets

In [ ]:

IMG_SIZE = 32

train_dirs = [
    "data/github_train_0",
    "data/github_train_1",
    "data/github_train_2",
    "data/github_train_3"
]

test_dir = "data/github_test"


def get_label_from_filename(filename):
    name = filename.lower()
    if "cat" in name or "gato" in name:
        return 0
    if "dog" in name or "perro" in name:
        return 1
    raise ValueError(f"No se puede inferir clase desde {filename}")


def load_images_from_directory(directory):
    X = []
    y = []
    filenames = []

    for file in os.listdir(directory):
        path = os.path.join(directory, file)
        try:
            img = Image.open(path).convert("RGB")
            img = img.resize((IMG_SIZE, IMG_SIZE))
            img = np.array(img) / 255.0

            label = get_label_from_filename(file)

            X.append(img)
            y.append(label)
            filenames.append(file)
        except:
            pass

    return np.array(X), np.array(y), filenames


X_list = []
y_list = []

for d in train_dirs:
    X_tmp, y_tmp, _ = load_images_from_directory(d)
    X_list.append(X_tmp)
    y_list.append(y_tmp)

X_train = np.concatenate(X_list)
y_train = np.concatenate(y_list)

X_test, y_test, test_filenames = load_images_from_directory(test_dir)

print("Train shape:", X_train.shape)
print("Test shape:", X_test.shape)


## 2. MiniEDA y visualización del dataset

In [ ]:

print("Distribución clases train:")
print(pd.Series(y_train).value_counts())


fig, axes = plt.subplots(3, 3, figsize=(6,6))

for i, ax in enumerate(axes.flat):
    ax.imshow(X_train[i])
    ax.set_title(f"Clase: {y_train[i]}")
    ax.axis("off")

plt.show()


## 3. Construcción del modelo CNN

In [ ]:

model = models.Sequential([
    
    layers.Conv2D(32, (3,3), activation='relu', input_shape=(32,32,3)),
    layers.MaxPooling2D((2,2)),
    
    layers.Conv2D(64, (3,3), activation='relu'),
    layers.MaxPooling2D((2,2)),
    
    layers.Conv2D(128, (3,3), activation='relu'),
    layers.Flatten(),
    
    layers.Dense(64, activation='relu'),
    layers.Dense(1, activation='sigmoid')
])

model.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['accuracy']
)

model.summary()


## 4. Entrenamiento con EarlyStopping

In [ ]:

early_stop = EarlyStopping(
    monitor='val_loss',
    patience=5,
    restore_best_weights=True
)

history = model.fit(
    X_train,
    y_train,
    epochs=50,
    batch_size=32,
    validation_split=0.2,
    callbacks=[early_stop]
)


## Historial de entrenamiento

In [ ]:

plt.figure()

plt.plot(history.history['loss'], label='train_loss')
plt.plot(history.history['val_loss'], label='val_loss')

plt.legend()
plt.title("Training history")
plt.show()


## 5. Evaluación del modelo

In [ ]:

pred_probs = model.predict(X_test).flatten()
pred_labels = (pred_probs > 0.5).astype(int)

print(classification_report(y_test, pred_labels))

cm = confusion_matrix(y_test, pred_labels)
print("Confusion matrix")
print(cm)


## 6. Selección de imágenes difíciles para CAPTCHA

In [ ]:

df_results = pd.DataFrame({
    "filename": test_filenames,
    "true_label": y_test,
    "pred_label": pred_labels,
    "prob_dog": pred_probs
})

df_results["prob_cat"] = 1 - df_results["prob_dog"]

misclassified = df_results[df_results.true_label != df_results.pred_label]

dogs_as_cats = misclassified[(misclassified.true_label == 1) & (misclassified.pred_label == 0)]
cats_as_dogs = misclassified[(misclassified.true_label == 0) & (misclassified.pred_label == 1)]

dogs_as_cats = dogs_as_cats.sort_values("prob_dog", ascending=False)
cats_as_dogs = cats_as_dogs.sort_values("prob_cat", ascending=False)

n_select_dogs = max(1, int(len(dogs_as_cats) * 0.1))
n_select_cats = max(1, int(len(cats_as_dogs) * 0.1))

hard_dogs = dogs_as_cats.head(n_select_dogs)
hard_cats = cats_as_dogs.head(n_select_cats)

hard_images = pd.concat([hard_dogs, hard_cats])

print("Imágenes seleccionadas como CAPTCHA difíciles:")
display(hard_images)



Las imágenes seleccionadas son aquellas que:

- El modelo **clasifica incorrectamente**
- Pero **con alta confianza en la clase equivocada**

Estas son precisamente las más complicadas para un sistema automático, por lo que son buenas candidatas para el CAPTCHA.
